Trains DistilBERT on sentences

Idea is from https://www.arxiv.org/abs/2509.20375

In [1]:
# Get dataset
from local_utilities.dataset import get_dataset_train, get_dataset_test, get_dataset_validation

# Train data
X_train, y_train = get_dataset_train()

# Validation data
X_val, y_val = get_dataset_validation()

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/home/tobias/.pyenv/versions/genai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create dataset class
from torch.utils.data import Dataset
import torch

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoder = self.tokenizer(
            self.texts[index],
            truncation = True,
            padding = "max_length",
            max_length = self.max_length,
            return_tensors = "pt"
        )

        item = {k: v.squeeze(0) for k, v in encoder.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)

        return item

In [3]:
# Prepare data for training
from local_utilities.model import get_distilbert_model
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(get_distilbert_model())
train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds = TextDataset(X_val, y_val, tokenizer)

In [4]:
# Create data loaders
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=96,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=96,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [5]:
# Create model
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    get_distilbert_model(),
    num_labels=2
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1531.64it/s, Materializing param=pre_classifier.weight]                                  


In [6]:
# Move model to GPU
from local_utilities.gpu import get_gpu

device = get_gpu()
model.to(device)
model = torch.compile(model, mode="max-autotune")

In [7]:
# Create optimizer
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=3e-5)

In [8]:
# Train model (1 epoch)
import torch

train_losses = []
val_losses = []

for epoch in range(3):
    # Train
    model.train()
    total_correct = 0.0    
    train_loss = 0.0

    for i, batch in enumerate(train_loader):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Input training data
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            # Reset gradients
            optimizer.zero_grad()
            
            # Compute loss
            outputs = model(**batch)
            loss = outputs.loss

        # Apply loss
        loss.backward()
        
        # Step forward
        optimizer.step()
        
        # Statistics
        train_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        total_correct += (preds == batch['labels']).sum().item()
        
    train_loss = train_loss / len(train_loader)
    train_accuracy = total_correct / len(train_loader.dataset)

    # Validate
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0

    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Compute loss
            outputs = model(**batch)
            loss = outputs.loss
            
            # Statistics
            val_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == batch["labels"]).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = correct / len(val_loader.dataset)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Show current outputs
    print(f"Epoch {epoch + 1}, train loss: {train_loss:.4f}, "
          f"train accuracy: {train_accuracy:.4f} | "
          f"val loss: {val_loss:.4f}, "
          f"val accuracy: {val_acc:.4f}")

Epoch 1, train loss: 0.1753, train accuracy: 0.9224 | val loss: 0.1622, val accuracy: 0.9277
Epoch 2, train loss: 0.1032, train accuracy: 0.9543 | val loss: 0.1639, val accuracy: 0.9369
Epoch 3, train loss: 0.0687, train accuracy: 0.9687 | val loss: 0.1864, val accuracy: 0.9379


In [9]:
"""
import matplotlib.pyplot as plt

x = [i+1 for i in range(len(train_losses))]

plt.figure(figsize=(8, 3))
plt.plot(x, train_losses, label="Train loss")
plt.plot(x, val_losses, label="Validation loss")
plt.legend()
plt.title("DistilBERT losses")
plt.ylabel("Loss")
plt.xlabel("Epochs")
plt.show()
"""

'\nimport matplotlib.pyplot as plt\n\nx = [i+1 for i in range(len(train_losses))]\n\nplt.figure(figsize=(8, 3))\nplt.plot(x, train_losses, label="Train loss")\nplt.plot(x, val_losses, label="Validation loss")\nplt.legend()\nplt.title("DistilBERT losses")\nplt.ylabel("Loss")\nplt.xlabel("Epochs")\nplt.show()\n'

In [10]:
# Save models
from local_utilities.directory import get_distilbert_directory

directory = get_distilbert_directory()
model.save_pretrained(directory)
tokenizer.save_pretrained(directory)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]


('./data/models/distilbert_sentences/tokenizer_config.json',
 './data/models/distilbert_sentences/tokenizer.json')